# Stochastic Calculus for Quantitative Finance

Stochastic calculus is the mathematical framework for modeling random processes in continuous time — the foundation of derivatives pricing.

## What You'll Learn
1. Random walks and Brownian motion
2. Geometric Brownian Motion (GBM)
3. Itô's Lemma — the chain rule for stochastic processes
4. Stochastic Differential Equations (SDEs)
5. The connection to Black-Scholes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Brownian Motion (Wiener Process)

A standard Brownian motion $W_t$ satisfies:
- $W_0 = 0$
- $W_t - W_s \sim N(0, t-s)$ for $t > s$
- Increments are independent
- Paths are continuous but **nowhere differentiable**

### Key Properties
- $E[W_t] = 0$
- $\text{Var}[W_t] = t$
- $dW_t \cdot dW_t = dt$ (the "Itô isometry")
- $dW_t \cdot dt = 0$

In [ ]:
# Simulate Brownian motion paths
T = 1.0      # 1 year
n_steps = 1000
dt = T / n_steps
n_paths = 5

t = np.linspace(0, T, n_steps + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Standard Brownian motion
for i in range(n_paths):
    dW = np.random.normal(0, np.sqrt(dt), n_steps)
    W = np.concatenate([[0], np.cumsum(dW)])
    axes[0].plot(t, W, alpha=0.7)

# Theoretical std envelope
axes[0].fill_between(t, -2*np.sqrt(t), 2*np.sqrt(t), alpha=0.1, color='gray', label='±2σ envelope')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('W(t)')
axes[0].set_title('Standard Brownian Motion')
axes[0].legend()

# Distribution of W(1) from many paths
W_T_samples = np.random.normal(0, np.sqrt(T), 10000)
axes[1].hist(W_T_samples, bins=60, density=True, alpha=0.6, color='steelblue')
x = np.linspace(-4, 4, 200)
axes[1].plot(x, norm.pdf(x, 0, np.sqrt(T)), 'r-', linewidth=2, label='N(0, T)')
axes[1].set_xlabel('W(T)')
axes[1].set_title(f'Distribution of W(T={T})')
axes[1].legend()
plt.tight_layout()
plt.show()

## 2. Geometric Brownian Motion (GBM)

The standard model for stock prices:

$$dS_t = \mu S_t \, dt + \sigma S_t \, dW_t$$

### Solution (via Itô's Lemma):

$$S_t = S_0 \exp\left[\left(\mu - \frac{\sigma^2}{2}\right)t + \sigma W_t\right]$$

Note the **Itô correction** term $-\frac{\sigma^2}{2}$: this is the difference between the arithmetic mean $\mu$ and the geometric mean.

In [ ]:
# Simulate GBM stock price paths
S0 = 100       # initial price
mu = 0.10      # 10% annual drift
sigma = 0.20   # 20% annual volatility
T = 1.0
n_steps = 252
dt = T / n_steps
n_paths = 1000

# Method 1: Exact solution
t = np.linspace(0, T, n_steps + 1)
Z = np.random.randn(n_paths, n_steps)
log_returns = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
log_prices = np.concatenate([np.zeros((n_paths, 1)), np.cumsum(log_returns, axis=1)], axis=1)
prices = S0 * np.exp(log_prices)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sample paths
for i in range(20):
    axes[0].plot(t, prices[i], alpha=0.4)
axes[0].plot(t, S0 * np.exp(mu * t), 'k--', linewidth=2, label=f'E[S(t)] = S₀·exp(μt)')
axes[0].set_xlabel('Time (years)')
axes[0].set_ylabel('Price')
axes[0].set_title('GBM Price Paths')
axes[0].legend()

# Terminal distribution
axes[1].hist(prices[:, -1], bins=60, density=True, alpha=0.6, color='steelblue')
axes[1].axvline(x=np.median(prices[:, -1]), color='red', linestyle='--',
                label=f'Median: {np.median(prices[:, -1]):.1f}')
axes[1].axvline(x=np.mean(prices[:, -1]), color='green', linestyle='--',
                label=f'Mean: {np.mean(prices[:, -1]):.1f}')
axes[1].set_xlabel('Terminal Price S(T)')
axes[1].set_title('Distribution of Terminal Price (Log-Normal)')
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Mean > Median because log-normal is right-skewed")
print(f"Median = S₀·exp((μ - σ²/2)·T) = {S0 * np.exp((mu - 0.5*sigma**2)*T):.2f}")

## 3. Itô's Lemma

The chain rule for stochastic calculus. If $X_t$ follows:
$$dX_t = \mu_t \, dt + \sigma_t \, dW_t$$

Then for $f(X_t, t)$:

$$df = \left(\frac{\partial f}{\partial t} + \mu_t \frac{\partial f}{\partial x} + \frac{1}{2}\sigma_t^2 \frac{\partial^2 f}{\partial x^2}\right)dt + \sigma_t \frac{\partial f}{\partial x}\, dW_t$$

The extra term $\frac{1}{2}\sigma_t^2 \frac{\partial^2 f}{\partial x^2}$ is the **Itô correction** — it exists because $(dW_t)^2 = dt \neq 0$.

In [ ]:
# Demonstration: Itô's Lemma applied to f(S) = ln(S)
# If dS = μS dt + σS dW, then by Itô's Lemma:
# d(ln S) = (μ - σ²/2) dt + σ dW

# Verify numerically
n_steps = 10000
dt = 1/252
S = [S0]
dW = np.random.normal(0, np.sqrt(dt), n_steps)

for i in range(n_steps):
    dS = mu * S[-1] * dt + sigma * S[-1] * dW[i]
    S.append(S[-1] + dS)
S = np.array(S)

# Compare actual log returns vs Itô prediction
log_returns_actual = np.diff(np.log(S))
ito_drift = mu - 0.5 * sigma**2

print(f"Itô's Lemma prediction: E[d(ln S)] / dt = μ - σ²/2 = {ito_drift:.4f}")
print(f"Empirical mean of d(ln S) / dt:          = {np.mean(log_returns_actual)/dt:.4f}")
print(f"\nWithout Itô correction (μ):               = {mu:.4f}")
print(f"→ The difference is the Itô correction: σ²/2 = {0.5*sigma**2:.4f}")

## 4. Ornstein-Uhlenbeck Process (Mean Reversion)

$$dX_t = \theta(\mu - X_t)\, dt + \sigma\, dW_t$$

- $\mu$: long-term mean
- $\theta$: speed of mean reversion
- $\sigma$: volatility

This is the model for **mean-reverting strategies** (pairs trading, stat arb).

In [ ]:
# Simulate Ornstein-Uhlenbeck process
theta = 5.0    # mean reversion speed (higher = faster reversion)
mu_ou = 0.0    # long-term mean
sigma_ou = 0.3
T = 2.0
n_steps = 1000
dt = T / n_steps
t = np.linspace(0, T, n_steps + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Different mean-reversion speeds
for theta_val in [0.5, 2.0, 5.0, 15.0]:
    X = [1.0]  # start away from mean
    for i in range(n_steps):
        dX = theta_val * (mu_ou - X[-1]) * dt + sigma_ou * np.random.normal(0, np.sqrt(dt))
        X.append(X[-1] + dX)
    axes[0].plot(t, X, label=f'θ = {theta_val}', alpha=0.8)

axes[0].axhline(y=mu_ou, color='black', linestyle='--', alpha=0.5, label='Long-term mean')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('X(t)')
axes[0].set_title('Ornstein-Uhlenbeck: Effect of Mean-Reversion Speed')
axes[0].legend(fontsize=8)

# Stationary distribution
n_samples = 10000
X_long = [0.0]
for i in range(n_samples):
    dX = theta * (mu_ou - X_long[-1]) * dt + sigma_ou * np.random.normal(0, np.sqrt(dt))
    X_long.append(X_long[-1] + dX)

# Theoretical stationary distribution: N(μ, σ²/(2θ))
stat_var = sigma_ou**2 / (2 * theta)
axes[1].hist(X_long[-5000:], bins=60, density=True, alpha=0.6, color='steelblue', label='Simulated')
x = np.linspace(-0.5, 0.5, 200)
axes[1].plot(x, norm.pdf(x, mu_ou, np.sqrt(stat_var)), 'r-', linewidth=2,
             label=f'N(μ, σ²/2θ) = N({mu_ou}, {stat_var:.4f})')
axes[1].set_title('Stationary Distribution')
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. From SDEs to Black-Scholes

Under risk-neutral pricing, the stock price follows:

$$dS_t = r S_t \, dt + \sigma S_t \, dW_t^Q$$

where $r$ is the risk-free rate and $W^Q$ is a Brownian motion under the risk-neutral measure.

Applying Itô's Lemma to the option price $V(S,t)$ and constructing a hedging portfolio leads to the **Black-Scholes PDE**:

$$\frac{\partial V}{\partial t} + rS\frac{\partial V}{\partial S} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} = rV$$

In [ ]:
# Black-Scholes formula (derived from stochastic calculus)
def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)

def black_scholes_put(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return K * np.exp(-r*T) * norm.cdf(-d2) - S * norm.cdf(-d1)

# Monte Carlo verification using risk-neutral pricing
S0 = 100
K = 105
T = 0.5
r = 0.05
sigma = 0.20

# Analytical
bs_call = black_scholes_call(S0, K, T, r, sigma)

# Monte Carlo under Q measure
n_mc = 100000
Z = np.random.randn(n_mc)
ST = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
payoffs = np.maximum(ST - K, 0)
mc_call = np.exp(-r*T) * np.mean(payoffs)
mc_se = np.exp(-r*T) * np.std(payoffs) / np.sqrt(n_mc)

print(f"Black-Scholes Call Price: {bs_call:.4f}")
print(f"Monte Carlo Call Price:   {mc_call:.4f} ± {1.96*mc_se:.4f}")
print(f"Difference:              {abs(bs_call - mc_call):.4f}")

## Exercises

1. **Brownian scaling**: Verify that $W_{ct}$ has the same distribution as $\sqrt{c} \cdot W_t$ by simulation.
2. **GBM calibration**: Given historical prices, estimate $\mu$ and $\sigma$ for GBM. Simulate future paths.
3. **Itô's Lemma**: Apply Itô's Lemma to $f(S) = S^2$. Verify numerically.
4. **OU parameter estimation**: Simulate an OU process and estimate $\theta$, $\mu$, $\sigma$ using OLS regression on $\Delta X_t = \theta(\mu - X_t)\Delta t + \sigma \epsilon_t$.
5. **Monte Carlo pricing**: Price a barrier option (knock-out call) using Monte Carlo under GBM.